## 03a_fact_claims — RCM Intelligence Platform
### Purpose
Builds the central fact table in the Gold star schema.
Joins Silver claims to all SCD dimensions using surrogate keys.
This is the table that powers all dashboards and KPI queries.

### What this does
1. Reads Silver claims (enriched inpatient data)
2. Joins to all four dimensions using surrogate keys
3. Adds CMS fiscal period from dim_date
4. Writes to Gold fact_claims Delta table
5. Optimizes for dashboard query performance

### Star schema
fact_claims
  ├── dim_hospital  (provider_id → hospital_key)
  ├── dim_provider  (provider_id → provider_key)
  ├── dim_drg       (drg_code → drg_code)
  ├── dim_payer     (drg_code → payer_key)
  └── dim_date      (batch_date → date_key)

### Inputs
- rcm_platform.rcm_silver.inpatient_claims_enriched
- rcm_platform.rcm_silver.dim_hospital
- rcm_platform.rcm_silver.dim_provider
- rcm_platform.rcm_silver.dim_drg
- rcm_platform.rcm_silver.dim_payer
- rcm_platform.rcm_silver.dim_date

### Outputs
- rcm_platform.rcm_gold.fact_claims

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Gold |
| Run order  | 6 — after 02b_scd_dimensions |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, round, expr,
    date_format, to_date, coalesce
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03a_fact_claims"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# STEP 1 — READ SILVER CLAIMS AND ALL DIMENSIONS
# ============================================================

print("=" * 55)
print("  READING SILVER AND DIMENSIONS")
print("=" * 55)

df_claims   = spark.table(TBL_SILVER_CLAIMS)
df_hospital = spark.table(TBL_DIM_HOSPITAL).filter("is_current = true")
df_provider = spark.table(TBL_DIM_PROVIDER).filter("is_current = true")
df_drg      = spark.table(TBL_DIM_DRG)
df_payer    = spark.table(TBL_DIM_PAYER).filter("is_current = true")
df_date     = spark.table(TBL_DIM_DATE)

print(f"Silver claims  : {df_claims.count():>8,}")
print(f"dim_hospital   : {df_hospital.count():>8,}")
print(f"dim_provider   : {df_provider.count():>8,}")
print(f"dim_drg        : {df_drg.count():>8,}")
print(f"dim_payer      : {df_payer.count():>8,}")
print(f"dim_date       : {df_date.count():>8,}")

In [0]:
# ============================================================
# STEP 2 — BUILD FACT TABLE WITH SURROGATE KEY JOINS
# Natural keys used for joining, surrogate keys stored in fact
# ============================================================

print("=" * 55)
print("  BUILDING FACT TABLE")
print("=" * 55)

# Prepare dimension lookups
df_hosp_lookup = df_hospital.select(
    col("provider_id"),
    col("hospital_key")
)

df_prov_lookup = df_provider.select(
    col("provider_id"),
    col("provider_key")
)

df_drg_lookup = df_drg.select(
    col("drg_code"),
    col("drg_severity_tier").alias("drg_severity_tier_dim")
)

df_payer_lookup = df_payer.select(
    col("drg_code"),
    col("payer_key"),
    col("contracted_rate"),
    col("coverage_pct")
)

# Get date key for batch date
batch_date_key = df_date \
    .filter(col("date_value") == lit(BATCH_DATE).cast("date")) \
    .select("date_key") \
    .collect()

date_key_val = batch_date_key[0]["date_key"] if batch_date_key else 20240101
print(f"Date key for batch : {date_key_val}")

# Build fact table
df_fact = df_claims \
    .join(df_hosp_lookup,  on="provider_id", how="left") \
    .join(df_prov_lookup,  on="provider_id", how="left") \
    .join(df_drg_lookup,   on="drg_code",    how="left") \
    .join(df_payer_lookup, on="drg_code",    how="left") \
    .select(
        # Surrogate keys
        coalesce(col("hospital_key"), lit("UNKNOWN")).alias("hospital_key"),
        coalesce(col("provider_key"), lit("UNKNOWN")).alias("provider_key"),
        coalesce(col("payer_key"),    lit("UNKNOWN")).alias("payer_key"),
        lit(date_key_val).alias("date_key"),

        # Natural keys
        col("provider_id"),
        col("drg_code"),

        # Denormalised dimensions
        col("provider_name"),
        col("provider_state"),
        col("provider_city"),
        col("provider_zip"),
        col("rural_urban_classification"),
        col("hospital_type"),
        col("hospital_ownership"),
        col("emergency_services"),
        col("hospital_overall_rating"),
        col("drg_description"),
        coalesce(
            col("drg_severity_tier_dim"),
            col("drg_severity_tier")
        ).alias("drg_severity_tier"),

        # Measures
        col("total_discharges"),
        col("avg_submitted_charge"),
        col("avg_total_payment"),
        col("avg_medicare_payment"),
        col("charge_to_payment_ratio"),
        col("medicare_payment_pct"),
        col("patient_responsibility"),
        col("revenue_gap"),
        col("revenue_gap_pct"),
        col("total_revenue_gap"),
        col("underpayment_flag"),
        col("high_volume_flag"),

        # Payer measures
        col("contracted_rate"),
        col("coverage_pct"),

        # Audit
        col("_batch_id"),
        col("_batch_date"),
        lit(BATCH_TIMESTAMP).cast("timestamp").alias("_gold_processed_at")
    )

fact_count = df_fact.count()
print(f"Fact table rows : {fact_count:,}")
print(f"Fact table cols : {len(df_fact.columns)}")

In [0]:
# ============================================================
# STEP 3 — WRITE FACT TABLE TO GOLD
# ============================================================

print("=" * 55)
print("  WRITING GOLD FACT TABLE")
print("=" * 55)

df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_FACT_CLAIMS)

# Add comment
spark.sql(f"""
    COMMENT ON TABLE {TBL_FACT_CLAIMS}
    IS 'Central fact table — Medicare inpatient claims with all 
    surrogate key joins to SCD dimensions. Powers all RCM dashboards.'
""")

# Optimize for dashboard queries
print("Running OPTIMIZE + ZORDER...")
spark.sql(f"""
    OPTIMIZE {TBL_FACT_CLAIMS}
    ZORDER BY (provider_state, drg_code, hospital_type)
""")

gold_rows = spark.table(TBL_FACT_CLAIMS).count()
print(f"\nGold fact table  : {TBL_FACT_CLAIMS}")
print(f"Rows written     : {gold_rows:,}")

print("\nSample fact rows:")
display(
    spark.table(TBL_FACT_CLAIMS).select(
        "hospital_key",
        "provider_key",
        "provider_id",
        "provider_name",
        "provider_state",
        "drg_code",
        "drg_description",
        "total_discharges",
        "avg_submitted_charge",
        "avg_medicare_payment",
        "charge_to_payment_ratio",
        "revenue_gap",
        "underpayment_flag"
    ).limit(5)
)

In [0]:
# ============================================================
# STEP 4 — VERIFICATION
# ============================================================

print("=" * 55)
print("  GOLD FACT TABLE VERIFICATION")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT provider_id)        AS unique_providers,
        COUNT(DISTINCT drg_code)           AS unique_drgs,
        COUNT(DISTINCT provider_state)     AS states_covered,
        COUNT(DISTINCT hospital_key)       AS hospital_keys_matched,
        COUNT(DISTINCT provider_key)       AS provider_keys_matched,
        COUNT(DISTINCT payer_key)          AS payer_keys_matched,
        SUM(underpayment_flag)             AS total_underpayment_flags,
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio,
        ROUND(SUM(total_revenue_gap), 0)   AS total_revenue_gap
    FROM {TBL_FACT_CLAIMS}
""").show(truncate=False)

print("\nSurrogate key coverage:")
spark.sql(f"""
    SELECT
        SUM(CASE WHEN hospital_key = 'UNKNOWN' THEN 1 ELSE 0 END) AS missing_hospital_key,
        SUM(CASE WHEN provider_key = 'UNKNOWN' THEN 1 ELSE 0 END) AS missing_provider_key,
        SUM(CASE WHEN payer_key    = 'UNKNOWN' THEN 1 ELSE 0 END) AS missing_payer_key,
        COUNT(*) AS total_rows
    FROM {TBL_FACT_CLAIMS}
""").show(truncate=False)

print("\nDelta history:")
spark.sql(f"""
    DESCRIBE HISTORY {TBL_FACT_CLAIMS} LIMIT 3
""").select("version", "timestamp", "operation", "operationMetrics") \
    .show(truncate=False)

In [0]:
# ============================================================
# STEP 5 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "gold",
    status           = "SUCCESS",
    rows_read        = fact_count,
    rows_written     = gold_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"fact_claims: {gold_rows:,} rows | "
        f"surrogate keys joined from 4 dimensions | "
        f"OPTIMIZE + ZORDER on provider_state, drg_code, hospital_type"
    )
)